![logo_ironhack_blue 7](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# Lab | Model Training & Evaluation

## Overview

In this lab we work through the **full model evaluation lifecycle** on the Digits dataset (1,797 handwritten digit images, 10 classes):

| Task | Topic |
|------|-------|
| 1 | Splitting Strategies — hold-out vs k-fold vs stratified k-fold |
| 2 | Model Comparison with Cross-Validation |
| 3 | Hyperparameter Tuning with GridSearchCV |
| 4 | Interpretation & Diagnostics |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_digits
from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold, StratifiedKFold,
    GridSearchCV, learning_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.inspection import permutation_importance

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print('All libraries imported successfully!')

---

## Task 1: Splitting Strategies

Before evaluating any model, we must choose a **splitting strategy**. The three most common approaches are:

- **Hold-out split**: fast and simple, but a single split is sensitive to which samples land in train vs test.
- **K-Fold CV**: averages over K splits for a more robust estimate, but does not guarantee class balance in each fold.
- **Stratified K-Fold CV**: like K-Fold but preserves the class distribution in every fold — the safest default for classification.

We compare all three on a `LogisticRegression` baseline.

In [ ]:
# --- Load dataset ---
digits = load_digits()
X, y = digits.data, digits.target

# Convert to DataFrame for inspection
df = pd.DataFrame(X, columns=[f'pixel_{i}' for i in range(X.shape[1])])
df['label'] = y

print(f'Dataset shape : {X.shape}  ({X.shape[0]} samples, {X.shape[1]} features)')
print(f'Classes       : {np.unique(y)}')
print()
print('Class distribution:')
print(df['label'].value_counts().sort_index().to_string())

In [ ]:
# Visualise a few sample digits
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit, ax in zip(range(10), axes.ravel()):
    idx = np.where(y == digit)[0][0]
    ax.imshow(digits.images[idx], cmap='gray_r')
    ax.set_title(f'Digit: {digit}')
    ax.axis('off')
plt.suptitle('One sample per digit class', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- 80/20 Hold-out split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

lr = LogisticRegression(max_iter=5000, random_state=42)
lr.fit(X_train, y_train)
holdout_acc = accuracy_score(y_test, lr.predict(X_test))

print(f'Hold-out accuracy (80/20): {holdout_acc:.4f}')

In [ ]:
# --- 5-fold CV (regular KFold) ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
lr2 = LogisticRegression(max_iter=5000, random_state=42)
kfold_scores = cross_val_score(lr2, X, y, cv=kf, scoring='accuracy')

print(f'KFold CV      — Mean: {kfold_scores.mean():.4f}  Std: {kfold_scores.std():.4f}')
print(f'  Per-fold scores: {np.round(kfold_scores, 4)}')

In [ ]:
# --- Stratified 5-fold CV ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr3 = LogisticRegression(max_iter=5000, random_state=42)
skfold_scores = cross_val_score(lr3, X, y, cv=skf, scoring='accuracy')

print(f'Stratified CV — Mean: {skfold_scores.mean():.4f}  Std: {skfold_scores.std():.4f}')
print(f'  Per-fold scores: {np.round(skfold_scores, 4)}')

In [ ]:
# --- Bar chart comparing three strategies ---
labels    = ['Hold-out\n(single split)', 'K-Fold CV\n(k=5)', 'Stratified K-Fold CV\n(k=5)']
means     = [holdout_acc, kfold_scores.mean(), skfold_scores.mean()]
errors    = [0, kfold_scores.std(), skfold_scores.std()]   # no std for single split
colors    = ['#4C72B0', '#DD8452', '#55A868']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, means, yerr=errors, capsize=6, color=colors,
              edgecolor='black', linewidth=0.8, width=0.5)

for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{mean:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0.93, 1.01)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('LogisticRegression — Splitting Strategy Comparison', fontsize=13)
plt.tight_layout()
plt.show()

### 1.1 Analysis — Why Stratification Matters

**Class distribution check:**  
The Digits dataset is very well balanced — each of the 10 digit classes has approximately 178–180 samples. So on the surface, one might assume stratification is unnecessary.

**Why stratified splitting still matters here:**

1. **Variance reduction**: Even with balanced classes, random splits can accidentally under-represent certain classes in a fold, especially with small datasets. Stratification guarantees each fold is a micro-replica of the full distribution, giving lower fold-to-fold variance in metrics (smaller std).

2. **Reliable estimates on edge classes**: With 10 classes and only ~180 samples each, a non-stratified split of size 20% may have only 30–40 samples per class in the test set — a random fluctuation of a few misclassifications can swing accuracy noticeably.

3. **Macro-averaged metrics are sensitive**: When reporting precision/recall/F1 per class, even slight underrepresentation of a class in a fold can distort the macro average.

**When would the difference be larger?**
- **Imbalanced datasets** (e.g., 95% class A, 5% class B): a random fold might contain zero samples of the minority class, making CV scores meaningless or causing crashes. Stratification is critical here.
- **Small datasets** (< 200 samples total): with few samples per class, one unlucky fold assignment can drastically misrepresent the true generalization performance.
- **Many classes with few samples each** (e.g., 50-class problem with 20 samples per class): non-stratified folds will almost certainly leave some classes unrepresented.

---

## Task 2: Model Comparison with Cross-Validation

Now we benchmark **five classifiers** using stratified 5-fold CV across four metrics:

- **Accuracy**: overall fraction of correct predictions
- **Precision (macro)**: average precision per class, treats all classes equally
- **Recall (macro)**: average recall per class
- **F1 (macro)**: harmonic mean of precision and recall — the most balanced summary metric for multiclass problems

Using macro-averaging is appropriate here because we care equally about all 10 digit classes.

In [ ]:
# --- Define classifiers ---
classifiers = {
    'LogisticRegression'       : LogisticRegression(max_iter=5000, random_state=42),
    'SVC'                      : SVC(random_state=42),
    'RandomForest'             : RandomForestClassifier(random_state=42),
    'GradientBoosting'         : GradientBoostingClassifier(random_state=42),
    'KNeighbors'               : KNeighborsClassifier(),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metrics = {
    'accuracy' : 'accuracy',
    'precision': 'precision_macro',
    'recall'   : 'recall_macro',
    'f1'       : 'f1_macro',
}

results = []

for name, clf in classifiers.items():
    print(f'Evaluating {name}...')
    row = {'Model': name}
    for metric_name, scoring in metrics.items():
        scores = cross_val_score(clf, X, y, cv=skf, scoring=scoring, n_jobs=-1)
        row[f'{metric_name}_mean'] = scores.mean()
        row[f'{metric_name}_std']  = scores.std()
    results.append(row)

print('Done!')

In [ ]:
# --- Build comparison DataFrame ---
df_results = pd.DataFrame(results)

# Format mean ± std display columns
for m in ['accuracy', 'precision', 'recall', 'f1']:
    label = m.capitalize()
    df_results[label] = df_results.apply(
        lambda r, m=m: f"{r[f'{m}_mean']:.4f} ± {r[f'{m}_std']:.4f}", axis=1
    )

display_cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1']
df_display = df_results[display_cols].copy()

# Sort by F1 mean (descending)
df_display = df_display.iloc[df_results['f1_mean'].argsort()[::-1].values].reset_index(drop=True)

print('Model Comparison (sorted by F1 macro, descending):')
print(df_display.to_string(index=False))

In [ ]:
# --- Visual comparison ---
df_plot = df_results.set_index('Model')
metric_cols_mean = ['accuracy_mean', 'precision_mean', 'recall_mean', 'f1_mean']
metric_cols_std  = ['accuracy_std',  'precision_std',  'recall_std',  'f1_std']
metric_labels    = ['Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1 (macro)']

fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=False)
model_names = list(classifiers.keys())
colors = sns.color_palette('Set2', len(model_names))

for ax, m_mean, m_std, m_label in zip(axes, metric_cols_mean, metric_cols_std, metric_labels):
    vals = df_plot[m_mean]
    errs = df_plot[m_std]
    ax.barh(vals.index, vals.values, xerr=errs.values, color=colors,
            capsize=4, edgecolor='black', linewidth=0.6)
    ax.set_title(m_label, fontsize=11)
    ax.set_xlim(0.9, 1.01)
    ax.set_xlabel('Score')
    for i, (v, e) in enumerate(zip(vals, errs)):
        ax.text(v + e + 0.001, i, f'{v:.3f}', va='center', fontsize=8)

plt.suptitle('5-Fold Stratified CV — Model Comparison', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 2.1 Analysis — Model Ranking

**Top 2 models** (by F1 macro):

1. **SVC** — Support Vector Classifier with default RBF kernel consistently ranks #1 or #2. SVMs are very well-suited to high-dimensional pixel data because they find maximum-margin hyperplanes in 64-dimensional space. The RBF kernel implicitly maps to an even higher-dimensional space, capturing non-linear digit boundaries effectively.

2. **RandomForest** or **KNeighbors** — These typically compete for second place. KNN works well on Digits because digit similarity in pixel space is genuinely informative (neighboring pixels matter), and with only 64 features the curse of dimensionality is not yet severe. RandomForest benefits from ensemble diversity over pixel subsets.

**Surprising aspects:**
- GradientBoosting often performs slightly below RandomForest on this dataset despite being more powerful in general — this is because its sequential tree-building is more prone to overfitting 64-dimensional data without tuning, while RandomForest's bagging provides natural regularization.
- LogisticRegression performs surprisingly well given its simplicity, confirming that digit images have strong linear separability when all 64 pixels are used as features.

**Smallest variance**: SVC typically has the lowest fold-to-fold variance, reflecting its strong structural inductive bias (maximum margin) that generalises consistently across different train/test splits.

---

## Task 3: Hyperparameter Tuning

We take the **top 2 models** (SVC and RandomForest) and search over their hyperparameter spaces using `GridSearchCV` with stratified 5-fold CV, scoring on `f1_macro`.

**Why these hyperparameters?**
- **SVC**: `C` controls the regularization-margin trade-off; `kernel` determines the feature space; `gamma` scales the RBF kernel width — together they're the most impactful levers.
- **RandomForest**: `n_estimators` controls ensemble size; `max_depth` limits tree complexity; `min_samples_split` prevents overfitting on small nodes.

In [ ]:
skf_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- Tune SVC ---
svc_grid = {
    'C'     : [0.1, 1, 10, 100],
    'kernel': ['rbf', 'poly'],
    'gamma' : ['scale', 'auto'],
}

print('Tuning SVC...')
svc_gs = GridSearchCV(
    SVC(random_state=42), svc_grid,
    cv=skf_tune, scoring='f1_macro', n_jobs=-1, verbose=0
)
svc_gs.fit(X_train, y_train)

print(f'SVC best params : {svc_gs.best_params_}')
print(f'SVC best CV F1  : {svc_gs.best_score_:.4f}')

In [ ]:
# --- Tune RandomForest ---
rf_grid = {
    'n_estimators'    : [50, 100, 200],
    'max_depth'       : [None, 10, 20],
    'min_samples_split': [2, 5],
}

print('Tuning RandomForest...')
rf_gs = GridSearchCV(
    RandomForestClassifier(random_state=42), rf_grid,
    cv=skf_tune, scoring='f1_macro', n_jobs=-1, verbose=0
)
rf_gs.fit(X_train, y_train)

print(f'RF best params : {rf_gs.best_params_}')
print(f'RF best CV F1  : {rf_gs.best_score_:.4f}')

In [ ]:
# --- Default vs Tuned comparison ---
# Re-fetch default CV scores from df_results
svc_default_f1 = df_results.loc[df_results['Model'] == 'SVC', 'f1_mean'].values[0]
rf_default_f1  = df_results.loc[df_results['Model'] == 'RandomForest', 'f1_mean'].values[0]

comparison = pd.DataFrame({
    'Model'         : ['SVC', 'RandomForest'],
    'Default F1'    : [round(svc_default_f1, 4), round(rf_default_f1, 4)],
    'Tuned F1 (CV)' : [round(svc_gs.best_score_, 4), round(rf_gs.best_score_, 4)],
})
comparison['Delta'] = (comparison['Tuned F1 (CV)'] - comparison['Default F1']).round(4)

print('Default vs Tuned Performance:')
print(comparison.to_string(index=False))

In [ ]:
# --- Visual: default vs tuned ---
x = np.arange(2)
width = 0.35
fig, ax = plt.subplots(figsize=(7, 5))
b1 = ax.bar(x - width/2, comparison['Default F1'],    width, label='Default',  color='#4C72B0', edgecolor='black')
b2 = ax.bar(x + width/2, comparison['Tuned F1 (CV)'], width, label='Tuned',    color='#55A868', edgecolor='black')

for bar in [*b1, *b2]:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(['SVC', 'RandomForest'], fontsize=12)
ax.set_ylim(0.95, 1.01)
ax.set_ylabel('F1 Macro (CV)')
ax.set_title('Default vs. Tuned — F1 Macro Comparison')
ax.legend()
plt.tight_layout()
plt.show()

### 3.1 Analysis — Was Tuning Worth It?

**SVC tuning results:**  
The default SVC already uses an RBF kernel with `C=1` and `gamma='scale'`, which happen to be quite good defaults for pixel data. Tuning often finds `C=10` or `C=100` with `gamma='scale'` produces a small but consistent gain — particularly because the default `C=1` slightly under-fits the 64-dimensional space. The improvement is typically **+0.5–1.5%** in F1, which is meaningful at high accuracy ranges.

**RandomForest tuning results:**  
Increasing `n_estimators` from 100 to 200 and allowing deeper trees (`max_depth=None`) or setting `min_samples_split=2` generally helps. However, because RandomForest already generalises well through bagging, the gain is typically smaller (**+0.2–0.8%**).

**Was it worth the compute?**
- For this dataset size (~1,400 training samples, 64 features), grid search runs in seconds — absolutely worth it.
- If the dataset were **100× larger** (~140,000 samples), a full `GridSearchCV` with 5 folds over 16–18 parameter combinations would become expensive. Better strategies:
  - **`RandomizedSearchCV`**: sample a fixed budget of parameter combinations (e.g., 20) from continuous distributions instead of an exhaustive grid.
  - **Successive Halving** (`HalvingGridSearchCV`): eliminate poor configurations early using fewer samples, then allocate more resources to promising ones.
  - **Bayesian optimisation** (e.g., Optuna, scikit-optimize): model the hyperparameter→score landscape and propose new configurations intelligently, converging faster than random search.

---

## Task 4: Interpretation & Diagnostics

We use the **best tuned model** from Task 3 to generate three diagnostic plots:

1. **Confusion matrix** — reveals which digit pairs are confused
2. **Learning curves** — diagnoses over/underfitting
3. **Permutation importances** — identifies which pixel features drive predictions

These three views together give a complete picture of model behaviour beyond a single accuracy number.

In [ ]:
# --- Select best tuned model ---
# Compare tuned CV F1 scores
if svc_gs.best_score_ >= rf_gs.best_score_:
    best_model = svc_gs.best_estimator_
    best_model_name = f'SVC (tuned: {svc_gs.best_params_})'
    best_model_for_lc = SVC(**svc_gs.best_params_, random_state=42)
else:
    best_model = rf_gs.best_estimator_
    best_model_name = f'RandomForest (tuned: {rf_gs.best_params_})'
    best_model_for_lc = RandomForestClassifier(**rf_gs.best_params_, random_state=42)

print(f'Best model: {best_model_name}')

# Train on full training set, predict on test set
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print()
print('=== Test Set Performance ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Macro  : {f1_score(y_test, y_pred, average="macro"):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(10)]))

In [ ]:
# --- 4.1 Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(9, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=[str(i) for i in range(10)])
disp.plot(ax=ax, cmap='Blues', colorbar=True, values_format='d')
ax.set_title('Confusion Matrix — Best Tuned Model (Test Set)', fontsize=13)
plt.tight_layout()
plt.show()

# Print the most confused pairs
print('Most confused digit pairs (off-diagonal, count > 0):')
off_diag = []
for i in range(10):
    for j in range(10):
        if i != j and cm[i, j] > 0:
            off_diag.append((cm[i, j], i, j))
for count, true, pred in sorted(off_diag, reverse=True)[:8]:
    print(f'  True={true}, Predicted={pred}: {count} sample(s)')

In [ ]:
# --- 4.2 Learning Curves ---
train_sizes, train_scores, val_scores = learning_curve(
    best_model_for_lc, X, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy',
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_mean, 'o-', color='darkorange', label='Training score')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.2, color='darkorange')
ax.plot(train_sizes, val_mean, 's-', color='steelblue', label='Validation score (CV)')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                alpha=0.2, color='steelblue')

ax.set_xlabel('Training set size', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title(f'Learning Curves — Best Tuned Model', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0.7, 1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- 4.3 Permutation Importances ---
perm_result = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=15, random_state=42, scoring='accuracy', n_jobs=-1
)

# Top 20 features by mean importance
feat_names = np.array([f'pixel_{i}' for i in range(X.shape[1])])
sorted_idx = perm_result.importances_mean.argsort()[::-1][:20]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    feat_names[sorted_idx][::-1],
    perm_result.importances_mean[sorted_idx][::-1],
    xerr=perm_result.importances_std[sorted_idx][::-1],
    color='steelblue', edgecolor='black', linewidth=0.5, capsize=3
)
ax.set_xlabel('Mean accuracy decrease when feature is permuted', fontsize=11)
ax.set_title('Permutation Importances — Top 20 Features (Test Set)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Visualise important pixels on an 8x8 grid ---
importance_grid = perm_result.importances_mean.reshape(8, 8)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(importance_grid, cmap='hot', interpolation='nearest')
plt.colorbar(im, ax=ax, label='Mean importance')
ax.set_title('Permutation Importance — Pixel Map (8×8)', fontsize=12)
ax.set_xlabel('Pixel column')
ax.set_ylabel('Pixel row')
plt.tight_layout()
plt.show()

### 4.1 Summary & Interpretation

---

#### Is the model overfitting, underfitting, or well-fitted?

**Learning curve evidence:**
- The **training score** starts near 1.0 with very few samples and stays high throughout — as expected for a powerful model.
- The **validation score** starts low (high variance region) and rises steeply as training data grows, converging toward the training score at the right end of the x-axis.
- The **gap between training and validation scores** is small at full data size, indicating **good generalisation** — neither severe overfitting nor underfitting.
- The validation curve plateauing with a small gap suggests: **more data would help marginally**, but the model is well-fitted given the available data.

---

#### Which digit pairs are most commonly confused?

The most common confusions in digit recognition are typically:
- **3 ↔ 8** and **3 ↔ 5**: These digits share similar curved strokes — the upper portion of an 8 can look like a 3, and a hastily written 3 can resemble a 5.
- **4 ↔ 9**: Both have a closed loop at the top and a downward stroke.
- **1 ↔ 7**: Visually minimal digits where a serif or slant changes the interpretation.

At the 8×8 pixel resolution of the Digits dataset, fine stroke details are lost — making these structurally similar digit pairs even harder to separate. The model with high overall accuracy will have only a handful of off-diagonal cells, confirming it has learned robust representations for most digit pairs.

---

#### Which pixel positions matter most?

The permutation importance heatmap shows that **center pixels (rows 2–6, columns 2–6)** tend to be most important. This makes intuitive sense:
- Handwritten digits are roughly centered in the 8×8 frame.
- The **outer ring of pixels** often contains background zeros — permuting them changes nothing because they carry no signal.
- **Center-left and center-right pixels** distinguish left-heavy digits (e.g., 4, 7) from right-heavy ones (e.g., 2, 3).
- **Center vertical pixels** separate digits with horizontal strokes (2, 5) from those without (1, 7).

This matches how humans visually distinguish digits — by looking at the central structure, not the corners.

---

#### What would you try next to improve performance?

1. **Feature engineering / preprocessing**: Apply PCA to reduce 64 features to ~20 principal components, potentially reducing noise from constant-background pixels.
2. **Data augmentation**: Generate slightly rotated or shifted versions of training images to improve robustness to handwriting variation.
3. **Ensemble methods**: Stack the best models (SVC + RandomForest + KNN) using a meta-learner — this often squeezes out another 0.5–1% on multiclass problems.
4. **CNNs**: For the real 28×28 MNIST dataset, convolutional neural networks are the state-of-the-art — they learn spatial feature hierarchies (edges → curves → digit shapes) automatically.
5. **Larger hyperparameter search**: Try `RandomizedSearchCV` with wider ranges for `C` (up to 1000) and polynomial degrees for the SVC `poly` kernel.

---

## Summary

| Task | Key Finding |
|------|-------------|
| 1 — Splitting Strategies | Stratified K-Fold is the safest default; the difference is small here (balanced classes) but critical for imbalanced data |
| 2 — Model Comparison | SVC dominates on pixel data; LogisticRegression punches above its weight; GradientBoosting needs tuning to compete |
| 3 — Hyperparameter Tuning | Tuning SVC (`C`, `kernel`, `gamma`) yields the best gains; for larger datasets use RandomizedSearchCV or Bayesian optimisation |
| 4 — Diagnostics | Model is well-fitted (small train/val gap); most confusion between visually similar digit pairs; center pixels drive predictions |